# DS Agent Backend Evaluation Notebook

This notebook evaluates the DS Agent backend (Node.js + Claude + Python execution) using the same benchmark framework as the LLM evaluation.

**Key Features:**
- Calls the DS Agent backend API (upload + chat)
- Handles multi-sheet Excel files with OpenAI classification
- Tracks inference time and cost estimates
- Outputs in compatible format for accuracy computation

**Prerequisites:**
- Node.js server running on `http://localhost:8000`
- Environment variables configured (JWT_TOKEN, OPENAI_API_KEY, etc.)
- Database and vector store configured (PostgreSQL, Qdrant)

In [1]:
import os
import json
import time
import requests
import uuid
from datetime import datetime
from typing import Dict, List, Any, Tuple, Optional
from tqdm.notebook import tqdm
from dotenv import load_dotenv
import openai

# Load environment variables
load_dotenv()

print("✓ Imports loaded")

✓ Imports loaded


## Configuration

In [2]:
# ========== CONFIGURATION ==========

# Backend API configuration
API_BASE_URL = "http://localhost:8000"

# Thread ID for conversation tracking
THREAD_ID = str(uuid.uuid4())  # Generate unique thread ID for this evaluation

# JWT token for authentication
JWT_TOKEN = os.getenv("JWT_TOKEN")

# OpenAI for sheet classification
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")

# Data configuration
# DATA_TAG = "data_subset_olive"  # Change to your data folder name
DATA_TAG = "data_sample"  # Change to your data folder name
DATA_PATH = f"./{DATA_TAG}/"
DATA_JSON_FILE = f"./{DATA_TAG}.json"

# Model name for tracking (the backend uses Claude Sonnet 4 internally)
MODEL_NAME = "dsagent-backend-v1"  # Can customize this for different backend versions

# Output configuration
AUTO_RESUME = False  # Set to True to automatically resume from most recent run

# Cost estimation (rough estimates - backend uses multiple models)
# These are APPROXIMATIONS for budgeting purposes
ESTIMATED_COST_PER_QUESTION = 0.15  # Rough estimate: $0.15 per question

# ===================================

if not JWT_TOKEN:
    raise ValueError("JWT_TOKEN not found in environment. Please set it in .env file")

if not OPENAI_API_KEY:
    raise ValueError("OPENAI_API_KEY not found in environment. Please set it in .env file")

print(f"Configuration loaded:")
print(f"  API Base URL: {API_BASE_URL}")
print(f"  Thread ID: {THREAD_ID}")
print(f"  Model: {MODEL_NAME}")
print(f"  Data path: {DATA_PATH}")
print(f"  JWT Token: {JWT_TOKEN[:20]}..." if JWT_TOKEN else "  JWT Token: Not set")

Configuration loaded:
  API Base URL: http://localhost:8000
  Thread ID: 258104ff-fc57-4d10-a8f2-802bf270a02e
  Model: dsagent-backend-v1
  Data path: ./data_sample/
  JWT Token: eyJhbGciOiJSUzI1NiIs...


## DS Agent API Functions

These functions are adapted from `example_query_v2.py` to interact with the backend.

In [3]:
def upload_file(file_path: str, jwt_token: str) -> Optional[Dict]:
    """
    Upload CSV/Excel file to the server.
    Returns dataInfo and gcsKey.
    """
    try:
        url = f"{API_BASE_URL}/api/upload-to-gcs"
        headers = {"Authorization": f"Bearer {jwt_token}"}

        with open(file_path, "rb") as f:
            files = {"csvFile": f}
            response = requests.post(url, files=files, headers=headers, timeout=120)

        if response.status_code == 200:
            result = response.json()
            dataInfo = result.get("dataInfo") or result.get("multiSheetData")
            gcsKey = result.get("gcsKey")
            return {"dataInfo": dataInfo, "gcsKey": gcsKey}
        else:
            print(f"✗ Upload failed: {response.status_code} {response.text}")
            return None

    except Exception as e:
        print(f"✗ Error uploading file: {e}")
        return None


def classify_sheets_with_llm(sheets: List[Dict], user_query: str) -> Tuple[List[int], bool]:
    """
    Use OpenAI to classify which sheets are relevant to the user query.
    Returns (selected_indices, is_universal).
    """
    options = []
    for i, sheet in enumerate(sheets):
        options.append({
            "index": i,
            "name": sheet.get('sheetName', sheet.get('sheet_name', f'Sheet {i}')),
            "summary": sheet.get('fileSummary', f"Sheet with {sheet.get('rowCount', '?')} rows")
        })

    system_prompt = "\n".join([
        "You are a classifier that picks which files a question refers to based on file summaries.",
        "You MUST respond with valid JSON only.",
        'The JSON structure must be: { "selected": [array of numbers], "universal": boolean }'
    ])

    summaries = "\n\n".join([
        f"Index: {o['index']}\nName: {o['name']}\nSummary: {o['summary']}"
        for o in options
    ])

    user_content = f"Question:\n{user_query}\n\nFiles:\n{summaries}\n\nRespond with JSON only:"

    try:
        client = openai.OpenAI(api_key=OPENAI_API_KEY)
        response = client.chat.completions.create(
            model="gpt-4o-mini",
            temperature=0,
            top_p=1,
            max_tokens=100,
            response_format={"type": "json_object"},
            messages=[
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": user_content}
            ]
        )

        result = json.loads(response.choices[0].message.content)
        selected = result.get("selected", [0])
        universal = result.get("universal", False)

        valid_selected = [i for i in selected if 0 <= i < len(sheets)]
        if not valid_selected:
            valid_selected = [0]

        return valid_selected, universal

    except Exception as e:
        print(f"⚠️ Error calling OpenAI for sheet classification: {e}")
        return [0], False


def send_chat_request_with_sse(
    message: str,
    data_info: Dict,
    jwt_token: str,
    thread_id: str,
    max_retries: int = 3
) -> Dict[str, Any]:
    """
    Send chat request to DS Agent backend with SSE streaming.
    Returns structured result with R (results), S (steps/code), N (natural language).
    """
    url = f"{API_BASE_URL}/api/chat"
    headers = {
        "Authorization": f"Bearer {jwt_token}",
        "Accept": "text/event-stream",
        "Content-Type": "application/json"
    }

    payload = {
        "message": message,
        "dataInfo": data_info,
        "multipleSnippets": True,
        "previousMessages": [],
        "thread_id": thread_id
    }

    result = {
        "R": {"outputs": "", "plots": [], "plot_count": 0},
        "S": {"concatenated_code": "", "individual_steps": []},
        "N": {"step_explanations": [], "final_interpretation": "", "reasoning_combined": ""},
        "metadata": {
            "timestamp": datetime.now().isoformat(),
            "total_steps": 0,
            "completion_id": "",
            "status": "failed",
            "time": 0
        }
    }

    for attempt in range(max_retries):
        try:
            start_time = time.time()
            response = requests.post(
                url,
                json=payload,
                headers=headers,
                stream=True,
                timeout=300
            )
            response.raise_for_status()

            current_step_explanations = {}
            event_type = None

            # Process SSE stream
            for line in response.iter_lines(decode_unicode=True):
                if line.startswith('event: '):
                    event_type = line[7:]
                elif line.startswith('data: '):
                    try:
                        data = json.loads(line[6:])
                        _process_sse_event(event_type, data, result, current_step_explanations)
                    except json.JSONDecodeError:
                        continue

            # Finalize
            _finalize_result(result, current_step_explanations)
            result["metadata"]["status"] = "success"
            result["metadata"]["time"] = time.time() - start_time

            return result

        except requests.exceptions.Timeout:
            if attempt < max_retries - 1:
                print(f"  ⚠️ Timeout (attempt {attempt + 1}), retrying...")
                time.sleep(5)
                continue
        except Exception as e:
            if attempt < max_retries - 1:
                print(f"  ⚠️ Error (attempt {attempt + 1}): {str(e)[:60]}, retrying...")
                time.sleep(2 ** attempt)
                continue
            result["metadata"]["error"] = str(e)

    result["metadata"]["status"] = "failed"
    return result


def _process_sse_event(event_type: str, data: Dict, result: Dict, current_step_explanations: Dict):
    """Process individual SSE events."""
    if event_type == "step":
        content = data.get("content", {})
        step_number = content.get("number", 0)

        if "code" in content and content["code"]:
            while len(result["S"]["individual_steps"]) <= step_number:
                result["S"]["individual_steps"].append("")
            result["S"]["individual_steps"][step_number] = content["code"]
            result["S"]["concatenated_code"] = "\n\n".join([
                step for step in result["S"]["individual_steps"] if step
            ])

        if "explanation" in content and content["explanation"]:
            current_step_explanations[step_number] = content["explanation"]

        if "output" in content and content["output"]:
            result["R"]["outputs"] += content["output"]

        if "plots" in content and content["plots"]:
            result["R"]["plots"].extend(content["plots"])

    elif event_type == "output":
        result["R"]["outputs"] = data.get("content", "")

    elif event_type == "plots":
        result["R"]["plots"] = data.get("content", [])
        result["R"]["plot_count"] = data.get("count", 0)

    elif event_type == "message":
        result["N"]["final_interpretation"] += data.get("content", "")

    elif event_type == "done":
        result["metadata"]["completion_id"] = data.get("content", "")


def _finalize_result(result: Dict, current_step_explanations: Dict):
    """Finalize result structure after SSE stream ends."""
    max_step = max(current_step_explanations.keys()) if current_step_explanations else -1
    result["N"]["step_explanations"] = []
    for i in range(max_step + 1):
        explanation = current_step_explanations.get(i, "")
        result["N"]["step_explanations"].append(explanation)

    reasoning_parts = []
    for i, explanation in enumerate(result["N"]["step_explanations"]):
        if explanation:
            reasoning_parts.append(f"Step {i + 1}: {explanation}")

    if result["N"]["final_interpretation"]:
        reasoning_parts.append(f"Final: {result['N']['final_interpretation']}")

    result["N"]["reasoning_combined"] = "\n\n".join(reasoning_parts)
    result["metadata"]["total_steps"] = len([s for s in result["S"]["individual_steps"] if s])
    result["R"]["plot_count"] = len(result["R"]["plots"])


print("✓ DS Agent API functions loaded")

✓ DS Agent API functions loaded


## Utility Functions

In [4]:
def find_excel_files(directory):
    """Find Excel files in directory."""
    excel_files = [file for file in os.listdir(directory)
                   if (file.lower().endswith(('.xlsx', '.xlsb', '.xlsm'))
                       and not "answer" in file.lower()
                       and not file.startswith('~$'))]
    return excel_files if excel_files else None


def read_txt(path):
    """Read text file."""
    with open(path, "r", encoding="utf-8") as f:
        return f.read()


def convert_dsagent_to_eval_format(
    dsagent_result: Dict,
    sample_id: str,
    question_idx: int
) -> Dict:
    """
    Convert DS Agent response format to eval notebook format.
    
    DS Agent format:
    {
      "R": {"outputs": "...", "plots": [...]},
      "S": {"concatenated_code": "..."},
      "N": {"final_interpretation": "..."},
      "metadata": {...}
    }
    
    Eval format:
    {
      "id": "sample_id",
      "question_idx": 0,
      "model": "model_name",
      "input_tokens": 0,  # Not available from backend
      "output_tokens": 0,  # Not available from backend
      "cost": 0.0,  # Estimated
      "time": 2.5,
      "response": "The answer is...",
      "dsagent_outputs": "...",  # Raw outputs
      "dsagent_code": "...",  # Generated code
      "dsagent_plots": [...]  # Plot URLs
    }
    """
    # Use final interpretation as the main response
    # If empty, fall back to outputs
    response = dsagent_result["N"].get("final_interpretation", "")
    if not response:
        response = dsagent_result["R"].get("outputs", "")
    
    # Build comprehensive response combining interpretation and outputs
    full_response_parts = []
    if dsagent_result["N"].get("final_interpretation"):
        full_response_parts.append(dsagent_result["N"]["final_interpretation"])
    if dsagent_result["R"].get("outputs"):
        full_response_parts.append(f"\nData Output:\n{dsagent_result['R']['outputs']}")
    
    response = "\n\n".join(full_response_parts) if full_response_parts else "No response generated"
    
    return {
        "id": sample_id,
        "question_idx": question_idx,
        "model": MODEL_NAME,
        "input_tokens": 0,  # Backend doesn't expose token counts
        "output_tokens": 0,
        "cost": ESTIMATED_COST_PER_QUESTION,  # Rough estimate
        "time": dsagent_result["metadata"].get("time", 0),
        "response": response,
        # Additional DS Agent specific fields
        "dsagent_outputs": dsagent_result["R"].get("outputs", ""),
        "dsagent_code": dsagent_result["S"].get("concatenated_code", ""),
        "dsagent_plots": dsagent_result["R"].get("plots", []),
        "dsagent_steps": dsagent_result["metadata"].get("total_steps", 0)
    }


print("✓ Utility functions loaded")

✓ Utility functions loaded


In [5]:
import json

def save_execution_outputs(
    sample_id: str,
    question_idx: int,
    dsagent_result: Dict,
    timestamp_folder: str,
    question_text: str = ""
) -> str:
    """
    Save execution details to an 'outputs' folder for debugging.
    
    Creates structure:
    <timestamp_folder>/
        outputs/
            execution_log.json (appended with each call)
    
    Returns: Path to the execution log entry
    """
    # Create outputs folder
    outputs_folder = os.path.join(timestamp_folder, "outputs")
    os.makedirs(outputs_folder, exist_ok=True)
    
    # Create execution log entry
    execution_entry = {
        "sample_id": sample_id,
        "question_number": question_idx + 1,
        "question_text": question_text[:200] + "..." if len(question_text) > 200 else question_text,
        "code_generated": dsagent_result["S"].get("concatenated_code", ""),
        "code_steps": dsagent_result["S"].get("individual_steps", []),
        "output_final": dsagent_result["R"].get("outputs", ""),
        "interpretation": dsagent_result["N"].get("final_interpretation", ""),
        "reasoning": dsagent_result["N"].get("reasoning_combined", ""),
        "plot_count": len(dsagent_result["R"].get("plots", [])),
        "status": dsagent_result["metadata"].get("status", "unknown"),
        "timestamp": dsagent_result["metadata"].get("timestamp", ""),
        "execution_time_seconds": dsagent_result["metadata"].get("time", 0),
        "total_steps": dsagent_result["metadata"].get("total_steps", 0),
        "error": dsagent_result["metadata"].get("error", None)
    }
    
    # Append to execution log file
    log_file_path = os.path.join(outputs_folder, "execution_log.json")
    
    # Load existing log if it exists
    execution_log = []
    if os.path.exists(log_file_path):
        try:
            with open(log_file_path, 'r', encoding='utf-8') as f:
                execution_log = json.load(f)
        except:
            execution_log = []
    
    # Add new entry
    execution_log.append(execution_entry)
    
    # Save updated log
    with open(log_file_path, 'w', encoding='utf-8') as f:
        json.dump(execution_log, f, indent=2, ensure_ascii=False)
    
    return log_file_path


print("✓ Output saving functions loaded")

✓ Output saving functions loaded


## Load Samples

In [6]:
# Load samples from JSON file
samples = []

if not os.path.exists(DATA_JSON_FILE):
    print(f"❌ Data file not found: {DATA_JSON_FILE}")
    print(f"   Please ensure the data file exists or update DATA_JSON_FILE configuration")
else:
    with open(DATA_JSON_FILE, "r") as f:
        for line in f:
            samples.append(eval(line.strip()))

    print(f"✓ Loaded {len(samples)} samples from {DATA_JSON_FILE}")

✓ Loaded 2 samples from ./data_sample.json


## Folder Management & Resume

In [7]:
import glob

def find_existing_runs(base_path: str, model_name: str):
    """Find all existing evaluation runs for a model."""
    model_folder = os.path.join(base_path, model_name)
    
    if not os.path.exists(model_folder):
        return []
    
    subdirs = [d for d in os.listdir(model_folder) 
               if os.path.isdir(os.path.join(model_folder, d))]
    
    runs = []
    for subdir in subdirs:
        folder_path = os.path.join(model_folder, subdir)
        json_files = glob.glob(os.path.join(folder_path, "*.json"))
        
        sample_files = [f for f in json_files 
                       if os.path.basename(f) not in ['results.json', 'results_process.json']]
        
        total_samples = len(sample_files)
        total_questions = 0
        total_errors = 0
        
        for json_file in sample_files:
            with open(json_file, 'r') as f:
                lines = f.readlines()
                for line in lines:
                    try:
                        entry = json.loads(line.strip())
                        if "error" in entry:
                            total_errors += 1
                        else:
                            total_questions += 1
                    except:
                        pass
        
        ctime = os.path.getctime(folder_path)
        mtime = os.path.getmtime(folder_path)
        runs.append({
            "folder": folder_path,
            "folder_name": subdir,
            "samples": total_samples,
            "questions": total_questions,
            "errors": total_errors,
            "created": datetime.fromtimestamp(ctime),
            "modified": datetime.fromtimestamp(mtime),
        })
    
    runs.sort(key=lambda x: x["modified"], reverse=True)
    return runs


def display_runs(runs):
    """Display existing runs."""
    if not runs:
        print("No existing runs found.")
        return
    
    print("\n" + "="*80)
    print("EXISTING EVALUATION RUNS")
    print("="*80)
    
    for i, run in enumerate(runs, 1):
        print(f"\n{i}. {run['folder_name']}")
        print(f"   Created: {run['created'].strftime('%Y-%m-%d %H:%M:%S')}")
        print(f"   Modified: {run['modified'].strftime('%Y-%m-%d %H:%M:%S')}")
        print(f"   Samples: {run['samples']} | Questions: {run['questions']} | Errors: {run['errors']}")
        
        if run['errors'] > 0:
            print(f"   ⚠️  Status: {run['errors']} questions have errors (need re-run)")
        else:
            print(f"   ✅ Status: All questions successful")


# Folder selection logic
sanitized_model_name = MODEL_NAME.replace('/', '_').replace(':', '_')
base_path = "./save_process"

runs = find_existing_runs(base_path, sanitized_model_name)

if not runs:
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    save_path = os.path.join(base_path, sanitized_model_name, timestamp)
    os.makedirs(save_path, exist_ok=True)
    print(f"\n✨ No existing runs found. Creating new folder:")
    print(f"   {save_path}")
else:
    display_runs(runs)
    
    if AUTO_RESUME:
        most_recent = runs[0]
        save_path = most_recent['folder']
        print(f"\n🔄 AUTO-RESUMING from most recent run:")
        print(f"   {save_path}")
    else:
        print("\n" + "="*80)
        print("OPTIONS:")
        print("="*80)
        for i, run in enumerate(runs, 1):
            status = "⚠️  Has errors" if run['errors'] > 0 else "✅ Complete"
            print(f"{i}. Resume from: {run['folder_name']} ({status})")
        print(f"{len(runs) + 1}. Create NEW run (with timestamp)")
        print("="*80)
        
        choice = input(f"\nEnter choice (1-{len(runs) + 1}): ").strip()
        choice_num = int(choice)
        
        if 1 <= choice_num <= len(runs):
            selected = runs[choice_num - 1]
            save_path = selected['folder']
            print(f"\n🔄 RESUMING from: {save_path}")
        else:
            timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
            save_path = os.path.join(base_path, sanitized_model_name, timestamp)
            os.makedirs(save_path, exist_ok=True)
            print(f"\n✨ Creating NEW run: {save_path}")

print(f"\n{'='*80}")
print(f"📁 Evaluation folder ready: {save_path}")
print(f"{'='*80}")


EXISTING EVALUATION RUNS

1. 20260130_232304
   Created: 2026-01-30 23:23:04
   Modified: 2026-01-30 23:53:58
   Samples: 2 | Questions: 18 | Errors: 0
   ✅ Status: All questions successful

2. 20260129_220708
   Created: 2026-01-29 22:07:08
   Modified: 2026-01-30 17:16:34
   Samples: 22 | Questions: 236 | Errors: 0
   ✅ Status: All questions successful

OPTIONS:
1. Resume from: 20260130_232304 (✅ Complete)
2. Resume from: 20260129_220708 (✅ Complete)
3. Create NEW run (with timestamp)

✨ Creating NEW run: ./save_process\dsagent-backend-v1\20260130_235743

📁 Evaluation folder ready: ./save_process\dsagent-backend-v1\20260130_235743


## Main Evaluation Loop

In [ ]:
# Main evaluation loop
total_cost_this_run = 0
total_cost_timestamp = 0
total_samples = 0
total_questions = 0
total_time = 0

for sample_idx in tqdm(range(len(samples)), desc="Processing samples"):
    sample = samples[sample_idx]
    
    if len(sample["questions"]) == 0:
        continue
    
    # Check which questions are already processed
    output_file = os.path.join(save_path, f"{sample['id']}.json")
    processed_indices = set()
    existing_cost = 0
    existing_time = 0
    
    if os.path.exists(output_file):
        with open(output_file, "r") as f:
            for line in f:
                try:
                    entry = json.loads(line.strip())
                    if "error" not in entry:
                        processed_indices.add(entry["question_idx"])
                        existing_cost += entry.get("cost", 0)
                        existing_time += entry.get("time", 0)
                except:
                    pass
        
        total_cost_timestamp += existing_cost
        
        if len(processed_indices) >= len(sample["questions"]):
            print(f"Skipping {sample['id']} - all questions completed")
            continue
        else:
            print(f"\nResuming {sample['id']} - {len(processed_indices)}/{len(sample['questions'])} done")
    
    # Find Excel files in sample directory
    sample_dir = os.path.join(DATA_PATH, sample["id"])
    excel_files = find_excel_files(sample_dir)
    
    if not excel_files:
        print(f"⚠️ No Excel files found for {sample['id']}, skipping")
        continue
    
    # Upload file (use first Excel file)
    excel_path = os.path.join(sample_dir, excel_files[0])
    print(f"\nProcessing sample {sample['id']} ({sample_idx + 1}/{len(samples)})...")
    print(f"  Uploading {excel_files[0]}...")
    
    upload_result = upload_file(excel_path, JWT_TOKEN)
    if not upload_result:
        print(f"  ✗ Upload failed, skipping sample")
        continue
    
    dataInfo = upload_result["dataInfo"]
    gcsKey = upload_result.get("gcsKey")
    
    # Handle multi-sheet Excel files
    if isinstance(dataInfo, list) and len(dataInfo) > 1:
        print(f"  Multi-sheet file detected ({len(dataInfo)} sheets)")
        # For now, use first sheet (or could classify per question)
        dataInfo = dataInfo[0]
        if gcsKey:
            dataInfo['gcsKey'] = gcsKey
    elif isinstance(dataInfo, list):
        dataInfo = dataInfo[0]
        if gcsKey:
            dataInfo['gcsKey'] = gcsKey
    
    # Read introduction
    introduction = read_txt(os.path.join(sample_dir, "introduction.txt"))
    
    # Read questions
    questions = []
    for question_name in sample["questions"]:
        question_text = read_txt(os.path.join(sample_dir, f"{question_name}.txt"))
        questions.append(question_text)

    # Build base context
    base_text = ""
    base_text += f"The introduction is detailed as follows.\n{introduction}\n"

    # Track costs for this sample
    sample_new_cost = 0
    sample_new_questions = 0
    sample_new_time = 0
    
    # Open file for appending
    with open(output_file, "a") as save_file:
        for question_idx, question in enumerate(questions):
            # Skip if already processed
            if question_idx in processed_indices:
                print(f"  Question {question_idx + 1}/{len(questions)}: ✓ Already completed")
                continue
            
            # Build full prompt
            prompt = base_text + f"The questions are detailed as follows.\n{question}"
            
            # # Build full prompt
            # prompt = f"""
            #     #####
            #     INTRODUCTION:
            #     The introduction below provides context about the data in the Excel file you have access to. Use this context to help answer the question that follows.
            #     {introduction}\n  
            
            #     #####
            #     SPECIAL INSTRUCTIONS:
            #     The Final Answer should be:
            #     1. From one of the options provided, if it is a Multiple Choice question.
            #     2. A numerical value, if it is a Fill in the Blank question.

            #     ##### 
            #     # USER QUESTION:
            #     The Question is as follows: \n{question}
            #     """


            print(f"  Question {question_idx + 1}/{len(questions)}: ", end="", flush=True)
            
            try:
                # Call DS Agent backend
                dsagent_result = send_chat_request_with_sse(
                    message=prompt,
                    data_info=dataInfo,
                    jwt_token=JWT_TOKEN,
                    thread_id=THREAD_ID
                )
                
                if dsagent_result["metadata"]["status"] == "failed":
                    raise Exception(dsagent_result["metadata"].get("error", "Unknown error"))
                
                # Save execution outputs (plots + debug info)
                try:
                    save_execution_outputs(
                        sample_id=sample["id"],
                        question_idx=question_idx,
                        dsagent_result=dsagent_result,
                        timestamp_folder=save_path,
                        question_text=question
                    )
                except Exception as e:
                    print(f"\n    ⚠️ Error saving outputs: {e}")
                
                # Convert to eval format
                answer = convert_dsagent_to_eval_format(
                    dsagent_result,
                    sample["id"],
                    question_idx
                )
                
                # Update totals
                total_cost_this_run += answer["cost"]
                total_cost_timestamp += answer["cost"]
                total_questions += 1
                total_time += answer["time"]
                sample_new_cost += answer["cost"]
                sample_new_questions += 1
                sample_new_time += answer["time"]
                
                print(f"Time: {answer['time']:.1f}s | Steps: {answer['dsagent_steps']} | Cost: ${answer['cost']:.6f} | Total: ${total_cost_timestamp:.6f}")
                
                # Save immediately
                json.dump(answer, save_file)
                save_file.write("\n")
                save_file.flush()
                
            except Exception as e:
                print(f"❌ ERROR: {e}")
                error_answer = {
                    "id": sample["id"],
                    "question_idx": question_idx,
                    "error": str(e),
                    "model": MODEL_NAME
                }
                json.dump(error_answer, save_file)
                save_file.write("\n")
                save_file.flush()
                
                # Try to save error info to outputs
                try:
                    error_result = {
                        "R": {"outputs": "", "plots": []},
                        "S": {"concatenated_code": "", "individual_steps": []},
                        "N": {"final_interpretation": "", "reasoning_combined": ""},
                        "metadata": {
                            "status": "failed",
                            "error": str(e),
                            "timestamp": datetime.now().isoformat(),
                            "time": 0,
                            "total_steps": 0
                        }
                    }
                    save_execution_outputs(
                        sample_id=sample["id"],
                        question_idx=question_idx,
                        dsagent_result=error_result,
                        timestamp_folder=save_path,
                        question_text=question
                    )
                except:
                    pass
                
                continue
    
    total_samples += 1
    total_sample_cost = existing_cost + sample_new_cost
    total_sample_time = existing_time + sample_new_time
    
    print(f"\n{'='*80}")
    print(f"✅ Completed {sample['id']}: {len(questions)} questions")
    if existing_cost > 0:
        print(f"💰 Sample cost: ${total_sample_cost:.6f} (${existing_cost:.6f} existing + ${sample_new_cost:.6f} new)")
        print(f"⏱️ Sample time: {total_sample_time:.1f}s ({existing_time:.1f}s existing + {sample_new_time:.1f}s new)")
    else:
        print(f"💰 Sample cost: ${sample_new_cost:.6f}")
        print(f"⏱️ Sample time: {sample_new_time:.1f}s")
    print(f"💰 Timestamp total so far: ${total_cost_timestamp:.6f}")
    print(f"{'='*80}\n")

print(f"\n{'='*80}")
print(f"🎉 Evaluation complete!")
print(f"Model: {MODEL_NAME}")
print(f"Samples processed: {total_samples}")
print(f"Questions processed (new): {total_questions}")
print(f"")
print(f"💰 Total Cost for this run: ${total_cost_this_run:.6f}")
print(f"💰 Total Cost for timestamp (all runs): ${total_cost_timestamp:.6f}")
print(f"⏱️ Total Time: {total_time:.1f}s ({total_time/60:.1f}m)")
if total_questions > 0:
    print(f"")
    print(f"Average Cost per Question (new): ${total_cost_this_run/total_questions:.6f}")
    print(f"Average Time per Question (new): {total_time/total_questions:.1f}s")
print(f"")
print(f"Results saved to: {save_path}")
print(f"📁 Execution logs and plots saved to: {os.path.join(save_path, 'outputs')}")
print(f"{'='*80}")

Processing samples:   0%|          | 0/2 [00:00<?, ?it/s]


Processing sample 00000001 (1/2)...
  Uploading MO16-Round-1-Sec-2-Chip-Off-The-Old-Block.xlsx...
  Multi-sheet file detected (3 sheets)
  Question 1/13: Time: 52.9s | Steps: 1 | Cost: $0.150000 | Total: $0.150000
  Question 2/13: Time: 62.2s | Steps: 1 | Cost: $0.150000 | Total: $0.300000
  Question 3/13: Time: 50.4s | Steps: 1 | Cost: $0.150000 | Total: $0.450000
  Question 4/13: Time: 70.7s | Steps: 1 | Cost: $0.150000 | Total: $0.600000
  Question 5/13: Time: 70.9s | Steps: 1 | Cost: $0.150000 | Total: $0.750000
  Question 6/13: Time: 65.3s | Steps: 1 | Cost: $0.150000 | Total: $0.900000
  Question 7/13: Time: 68.2s | Steps: 1 | Cost: $0.150000 | Total: $1.050000
  Question 8/13: Time: 62.8s | Steps: 1 | Cost: $0.150000 | Total: $1.200000
  Question 9/13: Time: 83.6s | Steps: 1 | Cost: $0.150000 | Total: $1.350000
  Question 10/13: Time: 117.4s | Steps: 3 | Cost: $0.150000 | Total: $1.500000
  Question 11/13: Time: 83.0s | Steps: 1 | Cost: $0.150000 | Total: $1.650000
  Question 1

## Notes

### Output Format

The notebook outputs results in a format compatible with the eval framework:

```json
{
  "id": "sample_id",
  "question_idx": 0,
  "model": "dsagent-backend-v1",
  "response": "Combined interpretation and outputs",
  "time": 12.5,
  "cost": 0.15,
  "dsagent_outputs": "Raw data outputs",
  "dsagent_code": "Generated Python code",
  "dsagent_plots": ["plot1.png", "plot2.png"],
  "dsagent_steps": 3
}
```

### Outputs Folder Structure (NEW)

For debugging and analysis, the notebook creates an `outputs/` folder in each timestamp directory:

```
./save_process/dsagent-backend-v1/20260130_193406/
├── 00000001.json                          # Evaluation results
├── 00000002.json
├── outputs/                                # ← NEW: Debug outputs
│   ├── plots/                              # Saved plots
│   │   ├── 00000001_q0_plot1.png
│   │   ├── 00000001_q0_plot2.png
│   │   ├── 00000001_q1_plot1.png
│   │   └── ...
│   └── execution_log.json                  # Detailed execution log
```

#### execution_log.json Format

```json
[
  {
    "sample_id": "00000001",
    "question_number": 1,
    "question_text": "What is the average sales...",
    "code_generated": "import pandas as pd\n...",
    "code_steps": ["import pandas as pd", "df.mean()"],
    "output_final": "The average is 125.5",
    "interpretation": "Based on the analysis...",
    "reasoning": "Step 1: Load data\nStep 2: Calculate...",
    "plots_saved": ["00000001_q0_plot1.png"],
    "plot_count": 1,
    "status": "success",
    "timestamp": "2026-01-30T19:34:12.123456",
    "execution_time_seconds": 12.5,
    "total_steps": 3,
    "error": null
  }
]
```

**Use cases:**
- **Debug failures**: Check `execution_log.json` for errors and code that was executed
- **Verify outputs**: Compare `output_final` with ground truth answers
- **Analyze code**: Review `code_generated` and `code_steps` for quality assessment
- **Inspect plots**: View saved plots in `plots/` folder for visual verification

### Limitations

1. **Token counts**: Backend doesn't expose token usage, so `input_tokens` and `output_tokens` are set to 0
2. **Cost estimation**: Uses rough estimate (`ESTIMATED_COST_PER_QUESTION`) since backend uses multiple models
3. **Multi-sheet handling**: Currently uses first sheet for all questions (could be enhanced with per-question classification)
4. **Plot format**: Plots are saved as PNG from base64 data; if plot data is malformed, saving will fail gracefully

### Next Steps

After running this notebook:
1. Use your existing accuracy computation scripts with the generated JSON files
2. Check `outputs/execution_log.json` to debug unsuccessful runs
3. Review plots in `outputs/plots/` for visual verification
4. The `response` field contains the final answer in the same format as the LLM evaluations

## Notes OLD

### Output Format

The notebook outputs results in a format compatible with the eval framework:

```json
{
  "id": "sample_id",
  "question_idx": 0,
  "model": "dsagent-backend-v1",
  "response": "Combined interpretation and outputs",
  "time": 12.5,
  "cost": 0.15,
  "dsagent_outputs": "Raw data outputs",
  "dsagent_code": "Generated Python code",
  "dsagent_plots": ["plot1.png", "plot2.png"],
  "dsagent_steps": 3
}
```

### Limitations

1. **Token counts**: Backend doesn't expose token usage, so `input_tokens` and `output_tokens` are set to 0
2. **Cost estimation**: Uses rough estimate (`ESTIMATED_COST_PER_QUESTION`) since backend uses multiple models
3. **Multi-sheet handling**: Currently uses first sheet for all questions (could be enhanced with per-question classification)

### Next Steps

After running this notebook, use your existing accuracy computation scripts with the generated JSON files. The `response` field contains the final answer in the same format as the LLM evaluations.